# 02 — Hebrew Contextual Embeddings

Generates XLM-RoBERTa contextual embeddings for Hebrew content words.

## Strategy
1. **Sentence assignment via timestamps** — the full English transcript (`podcast_transcript.csv`)
   has all 5136 words with timestamps. All 1735 content words have exact matches there.
   We derive sentence time boundaries from it, then assign each content word to its sentence
   by timestamp lookup. NO Hebrew text matching needed for this step.
2. **Within-sentence matching via cross-lingual cosine similarity** — embed the target Hebrew
   word in isolation, embed the full Hebrew sentence, find the token span with highest cosine
   similarity to the isolated vector. No morphological rules, no language-specific heuristics.
3. **No fallback** — words below the similarity threshold are dropped. 

## Outputs
| File | Description |
|------|-------------|
| `he_contextual_aligned_embeddings.csv` | (N_he, 768) — quality matches only |
| `he_contextual_matched_indices.csv` | original word_idx for each kept row |
| `he_contextual_quality_flags.csv` | all 1735 words with similarity score and match status |

## 1. Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict
from tqdm import tqdm

print('Imports ready.')
print(f'PyTorch : {torch.__version__}  |  CUDA : {torch.cuda.is_available()}')

## 2. Config & Data Loading

In [ ]:
LANG           = 'he'
SIM_THRESHOLD  = 0.70   # cosine similarity — drop matches below this
DATA_DIR       = '../data/processed/'

# ── Word-level transcript (1735 content words, with timestamps + translations) ─
content_df = pd.read_csv(
    '../data/Amirim_Project_Submission/translated_podcast_transcript_filtered.csv'
)

# ── Full English transcript (5136 words, all with timestamps) ──────────────────
full_df = pd.read_csv(
    '../data/ds005574/stimuli/podcast_transcript.csv'
)  # columns: word, start, end

# ── Hebrew sentences (parallel to English, indexed 1..402) ────────────────────
import csv
he_sentences = []
with open('../data/sentences/podcast_sentences_he.csv', encoding='utf-8-sig') as f:
    for row in csv.reader(f):
        if row and row[0] != 'sentence_id':
            he_sentences.append(row[2])   # column 2 = sentence_he

# ── English sentences (for building time boundaries) ──────────────────────────
en_sentences = []
with open('../data/sentences/podcast_sentences_en.csv', encoding='utf-8-sig') as f:
    next(f)
    for line in f:
        line = line.rstrip('\n\r')
        if line:
            en_sentences.append(line[line.index(',')+1:])

assert len(he_sentences) == len(en_sentences), \
    f'Sentence count mismatch: HE={len(he_sentences)} EN={len(en_sentences)}'

print(f'Content words    : {len(content_df)}')
print(f'Full transcript  : {len(full_df)} words')
print(f'Sentences        : {len(he_sentences)}')
print(f'Similarity threshold : {SIM_THRESHOLD}')

## 3. Build Sentence Time Boundaries from Full Transcript

In [ ]:
def norm_en(text):
    """Normalize English word for matching: lowercase, alphanumeric only."""
    return re.sub(r'[^a-z0-9]', '', text.lower())

full_norm = [norm_en(w) for w in full_df['word'].tolist()]
n_full    = len(full_norm)

# For each sentence, find the time range in the full transcript
# by sequentially matching the first 2 unique words of each sentence.
sentence_boundaries = {}   # sent_idx (0-based) -> (start_time, end_time)
ptr = 0

for sent_idx, sentence in enumerate(en_sentences):
    sent_words = [norm_en(w) for w in sentence.split() if norm_en(w)]
    if not sent_words:
        continue

    # Search forward from ptr with generous lookahead
    found = False
    for offset in range(min(200, n_full - ptr)):
        start = ptr + offset
        if start >= n_full:
            break
        if full_norm[start] != sent_words[0]:
            continue

        # Match as many sentence words as possible from this start
        matched, fp = 0, start
        for sw in sent_words:
            for skip in range(5):   # allow up to 4 unmatched tokens between words
                if fp + skip < n_full and full_norm[fp + skip] == sw:
                    fp = fp + skip + 1
                    matched += 1
                    break

        if matched >= max(2, len(sent_words) * 0.65):
            end_idx = min(fp - 1, n_full - 1)
            sentence_boundaries[sent_idx] = (
                full_df.iloc[start]['start'],
                full_df.iloc[end_idx]['end'],
            )
            ptr = start
            found = True
            break

    if not found:
        # Global fallback: find first-word match anywhere
        for i in range(n_full):
            if full_norm[i] == sent_words[0]:
                end_i = min(i + len(sent_words), n_full - 1)
                sentence_boundaries[sent_idx] = (
                    full_df.iloc[i]['start'],
                    full_df.iloc[end_i]['end'],
                )
                break

print(f'Sentences with time boundaries : {len(sentence_boundaries)} / {len(en_sentences)}')

# Build sorted list for fast lookup
boundary_list = sorted(
    [(v[0], v[1], k) for k, v in sentence_boundaries.items()]
)  # (start_time, end_time, sent_idx)

## 4. Assign Each Content Word to Its Sentence via Timestamp

In [ ]:
def find_sentence_for_time(t, boundary_list):
    """Return sentence_idx whose time range contains t (or nearest if gap)."""
    best_si, best_dist = None, float('inf')
    for (s_start, s_end, si) in boundary_list:
        if s_start <= t <= s_end:
            return si          # exact hit
        dist = min(abs(t - s_start), abs(t - s_end))
        if dist < best_dist:
            best_dist, best_si = dist, si
    return best_si

word_to_sent = {}
for wi in range(len(content_df)):
    t  = content_df.iloc[wi]['start']
    si = find_sentence_for_time(t, boundary_list)
    word_to_sent[wi] = si

assigned = sum(1 for v in word_to_sent.values() if v is not None)
print(f'Sentence assignment : {assigned} / {len(content_df)} ({assigned/len(content_df)*100:.1f}%)')

# Verify first 10
print('\nFirst 10 assignments:')
for wi in range(10):
    si  = word_to_sent[wi]
    row = content_df.iloc[wi]
    preview = he_sentences[si][:55] if si is not None else 'UNASSIGNED'
    print(f"  [{wi:3d}] EN='{row['en']:12s}'  HE='{row['he']:12s}'  -> sent {si}: '{preview}'") 

## 5. Load XLM-RoBERTa

In [ ]:
print('Loading XLM-RoBERTa-base...')
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model     = AutoModel.from_pretrained('xlm-roberta-base')
model.eval()
print('Model ready.')

## 6. Embedding Helper Functions

In [ ]:
def get_token_embeddings(text):
    """
    Run text through XLM-RoBERTa and return per-word-token averaged embeddings.
    Returns: list of 768d numpy arrays, one per word token (subwords averaged).
    """
    if not isinstance(text, str) or not text.strip():
        return []
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.no_grad():
        out = model(**enc)
    hidden = out.last_hidden_state.squeeze(0)   # (seq_len, 768)
    word_ids = enc.word_ids()

    word_vecs = defaultdict(list)
    for idx, wid in enumerate(word_ids):
        if wid is not None:
            word_vecs[wid].append(hidden[idx].cpu().numpy())

    return [np.mean(vecs, axis=0) for wid, vecs in sorted(word_vecs.items())]


def get_isolated_embedding(target_text):
    """
    Embed the target word/phrase in isolation.
    Returns: single 768d vector (mean over all word tokens).
    """
    vecs = get_token_embeddings(target_text)
    if not vecs:
        return np.zeros(768, dtype=np.float32)
    return np.mean(vecs, axis=0).astype(np.float32)


def cosine_sim(a, b):
    a = a / (np.linalg.norm(a) + 1e-8)
    b = b / (np.linalg.norm(b) + 1e-8)
    return float(np.dot(a, b))


def find_best_span(isolated_vec, sent_token_vecs, n_target_tokens):
    """
    Find the n_target_tokens-word span in sent_token_vecs that has the
    highest cosine similarity to isolated_vec (which is the mean of the
    target's isolated token embeddings).

    Returns: (contextual_vec, best_similarity)
      contextual_vec — mean of the matched span's contextual embeddings
      best_similarity — cosine similarity of that span to isolated_vec
    """
    n = len(sent_token_vecs)
    if n == 0 or n_target_tokens > n:
        return None, 0.0

    best_sim   = -2.0
    best_start = 0

    for start in range(n - n_target_tokens + 1):
        span      = sent_token_vecs[start : start + n_target_tokens]
        span_mean = np.mean(span, axis=0)
        sim       = cosine_sim(isolated_vec, span_mean)
        if sim > best_sim:
            best_sim   = sim
            best_start = start

    span_mean = np.mean(
        sent_token_vecs[best_start : best_start + n_target_tokens], axis=0
    ).astype(np.float32)
    return span_mean, best_sim


print('Helper functions defined.')

## 7. Extraction Loop

In [ ]:
# Cache sentence embeddings so each sentence is tokenized only once
sent_embed_cache = {}

def get_sent_vecs(si):
    if si not in sent_embed_cache:
        sent_embed_cache[si] = get_token_embeddings(he_sentences[si])
    return sent_embed_cache[si]


results = []   # list of dicts: word_idx, vec, similarity, matched

for wi in tqdm(range(len(content_df)), desc='Words'):
    row        = content_df.iloc[wi]
    target_raw = str(row[LANG]).strip()    # e.g. 'בעלי חיים'
    si         = word_to_sent[wi]

    if si is None:
        results.append({'word_idx': wi, 'vec': None, 'similarity': 0.0, 'matched': False})
        continue

    # Number of words in the target (multi-word phrases split by space)
    n_target = len(target_raw.split())

    # Embed target in isolation
    isolated_vec = get_isolated_embedding(target_raw)

    # Get contextual token embeddings for the full sentence
    sent_vecs = get_sent_vecs(si)

    # Find the best matching span
    ctx_vec, sim = find_best_span(isolated_vec, sent_vecs, n_target)

    matched = (ctx_vec is not None) and (sim >= SIM_THRESHOLD)
    results.append({
        'word_idx'  : wi,
        'vec'       : ctx_vec if matched else None,
        'similarity': round(sim, 4),
        'matched'   : matched,
    })

print('\nExtraction complete.')

## 8. Alignment Report

In [ ]:
matched_results  = [r for r in results if r['matched']]
dropped_results  = [r for r in results if not r['matched']]
sims_all         = [r['similarity'] for r in results]
sims_matched     = [r['similarity'] for r in matched_results]

print('=' * 55)
print('HEBREW ALIGNMENT REPORT')
print('=' * 55)
print(f'Total words          : {len(results)}')
print(f'Matched (sim >= {SIM_THRESHOLD}) : {len(matched_results)}  ({len(matched_results)/len(results)*100:.1f}%)')
print(f'Dropped              : {len(dropped_results)}  ({len(dropped_results)/len(results)*100:.1f}%)')
print()
print(f'Similarity stats (all words):')
print(f'  mean  : {np.mean(sims_all):.3f}')
print(f'  median: {np.median(sims_all):.3f}')
print(f'  min   : {np.min(sims_all):.3f}')
print(f'  max   : {np.max(sims_all):.3f}')
print()
print(f'First 10 dropped words (low similarity):')
for r in sorted(dropped_results, key=lambda x: x['similarity'])[:10]:
    row = content_df.iloc[r['word_idx']]
    si  = word_to_sent[r['word_idx']]
    sent_preview = he_sentences[si][:50] if si is not None else 'no sentence'
    print(f"  [{r['word_idx']:4d}] sim={r['similarity']:.3f}  "
          f"EN='{row['en']}'  HE='{row['he']}'  sent='{sent_preview}'")

## 9. Save

In [ ]:
kept_indices = [r['word_idx'] for r in matched_results]
kept_vecs    = np.stack([r['vec'] for r in matched_results])   # (N_he, 768)

# ── Embeddings ────────────────────────────────────────────────────────────────
pd.DataFrame(kept_vecs).to_csv(
    DATA_DIR + 'he_contextual_aligned_embeddings.csv', index=False
)

# ── Matched index (analogous to en_contextual_matched_indices.csv) ─────────────
pd.DataFrame({'original_word_idx': kept_indices}).to_csv(
    DATA_DIR + 'he_contextual_matched_indices.csv', index=False
)

# ── Quality flags (all 1735 words, for analysis) ──────────────────────────────
quality_df = pd.DataFrame({
    'word_idx'   : [r['word_idx']   for r in results],
    'en'         : content_df['en'].tolist(),
    'he'         : content_df['he'].tolist(),
    'sentence_id': [word_to_sent[r['word_idx']] for r in results],
    'similarity' : [r['similarity']  for r in results],
    'matched'    : [r['matched']     for r in results],
})
quality_df.to_csv(DATA_DIR + 'he_contextual_quality_flags.csv', index=False)

print(f'Saved:')
print(f'  he_contextual_aligned_embeddings.csv  {kept_vecs.shape}')
print(f'  he_contextual_matched_indices.csv     {len(kept_indices)} rows')
print(f'  he_contextual_quality_flags.csv       {len(quality_df)} rows')
print(f'\nSanity checks:')
print(f'  Any NaN : {np.isnan(kept_vecs).any()}')
print(f'  Col var : {kept_vecs.var(axis=0).mean():.5f}')
print(f'\nNext step: run 03_Generate_Embeddings_AR.ipynb with the same strategy,')
print(f'then run notebooks 05 and 06.')